<a href="https://colab.research.google.com/github/TesterSim2/Focused-IDE/blob/main/Fusing%20Determinism%20and%20Probability/Fusing_(D%26P)v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, REPOSITORY MOCK, AND AST PARSER
# ==============================================================================

# 1. Install required dependencies
!pip install -q tree-sitter==0.22.3 tree-sitter-python==0.21.0 pydantic networkx faiss-cpu

import os
import json
import logging
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, asdict
from pathlib import Path

# Setup Production-Grade Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s[%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("AST_Parser")

# Import tree-sitter
import tree_sitter
import tree_sitter_python

# ==============================================================================
# DATA MODELS (Strict Typing)
# ==============================================================================
@dataclass
class ImportMeta:
    module: str
    names: List[str]
    is_from_import: bool

@dataclass
class FunctionMeta:
    name: str
    signature: str
    docstring: Optional[str]
    start_line: int
    end_line: int
    is_method: bool

@dataclass
class ClassMeta:
    name: str
    bases: List[str]
    docstring: Optional[str]
    methods: List[FunctionMeta]
    start_line: int
    end_line: int

@dataclass
class FileMeta:
    filepath: str
    imports: List[ImportMeta]
    classes: List[ClassMeta]
    functions: List[FunctionMeta]  # Module-level functions

# ==============================================================================
# AST ANALYZER CORE
# ==============================================================================
class CodebaseAnalyzer:
    """
    A production-ready AST parser that extracts architectural metadata from a
    codebase while discarding implementation bodies to optimize LLM context windows.
    """
    def __init__(self):
        # Modern tree-sitter instantiation
        self.language = tree_sitter.Language(tree_sitter_python.language())
        self.parser = tree_sitter.Parser(self.language)
        logger.info("CodebaseAnalyzer initialized with tree-sitter-python.")

    def _get_text(self, node: tree_sitter.Node, source_bytes: bytes) -> str:
        """Safely extracts text from a node."""
        if node is None:
            return ""
        return source_bytes[node.start_byte:node.end_byte].decode("utf8", errors="replace")

    def _extract_docstring(self, node: tree_sitter.Node, source_bytes: bytes) -> Optional[str]:
        """Extracts docstring if it is the first statement in a block."""
        if node.type in ('function_definition', 'class_definition'):
            body_node = node.child_by_field_name('body')
            if body_node and body_node.child_count > 0:
                first_statement = body_node.children[0]
                if first_statement.type == 'expression_statement':
                    string_node = first_statement.children[0]
                    if string_node.type == 'string':
                        return self._get_text(string_node, source_bytes).strip('"""\'\'\'')
        return None

    def _parse_imports(self, root_node: tree_sitter.Node, source_bytes: bytes) -> List[ImportMeta]:
        imports =[]
        for child in root_node.children:
            if child.type == 'import_statement':
                # e.g., import os, sys
                names =[self._get_text(n, source_bytes) for n in child.children if n.type == 'dotted_name']
                for name in names:
                    imports.append(ImportMeta(module=name, names=[], is_from_import=False))
            elif child.type == 'import_from_statement':
                # e.g., from app.models import User
                module_node = child.child_by_field_name('module_name')
                module_name = self._get_text(module_node, source_bytes) if module_node else ""
                names =[self._get_text(n, source_bytes) for n in child.children if n.type == 'dotted_name' or n.type == 'aliased_import']
                imports.append(ImportMeta(module=module_name, names=names, is_from_import=True))
        return imports

    def _parse_function(self, node: tree_sitter.Node, source_bytes: bytes, is_method: bool = False) -> FunctionMeta:
        name_node = node.child_by_field_name('name')
        params_node = node.child_by_field_name('parameters')
        return_node = node.child_by_field_name('return_type')

        name = self._get_text(name_node, source_bytes)
        params = self._get_text(params_node, source_bytes)
        ret_type = f" -> {self._get_text(return_node, source_bytes)}" if return_node else ""

        signature = f"def {name}{params}{ret_type}:"
        docstring = self._extract_docstring(node, source_bytes)

        return FunctionMeta(
            name=name,
            signature=signature,
            docstring=docstring,
            start_line=node.start_point[0] + 1,
            end_line=node.end_point[0] + 1,
            is_method=is_method
        )

    def _parse_class(self, node: tree_sitter.Node, source_bytes: bytes) -> ClassMeta:
        name_node = node.child_by_field_name('name')
        superclasses_node = node.child_by_field_name('superclasses')

        name = self._get_text(name_node, source_bytes)
        bases =[]
        if superclasses_node:
            bases =[self._get_text(n, source_bytes) for n in superclasses_node.children if n.type == 'identifier']

        docstring = self._extract_docstring(node, source_bytes)
        methods =[]

        body_node = node.child_by_field_name('body')
        if body_node:
            for child in body_node.children:
                if child.type == 'function_definition':
                    methods.append(self._parse_function(child, source_bytes, is_method=True))

        return ClassMeta(
            name=name,
            bases=bases,
            docstring=docstring,
            methods=methods,
            start_line=node.start_point[0] + 1,
            end_line=node.end_point[0] + 1
        )

    def analyze_file(self, filepath: str) -> Optional[FileMeta]:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                source_code = f.read()
            source_bytes = source_code.encode('utf8')

            tree = self.parser.parse(source_bytes)
            root_node = tree.root_node

            imports = self._parse_imports(root_node, source_bytes)
            classes =[]
            functions =[]

            for child in root_node.children:
                if child.type == 'class_definition':
                    classes.append(self._parse_class(child, source_bytes))
                elif child.type == 'function_definition':
                    functions.append(self._parse_function(child, source_bytes, is_method=False))

            return FileMeta(filepath=filepath, imports=imports, classes=classes, functions=functions)

        except Exception as e:
            logger.error(f"Failed to parse {filepath}: {str(e)}")
            return None

# ==============================================================================
# DUMMY REPOSITORY GENERATOR
# ==============================================================================
def create_dummy_repository(base_path: str):
    """Creates a realistic mini FastAPI project to test our analyzer."""
    repo_path = Path(base_path)
    repo_path.mkdir(parents=True, exist_ok=True)

    files = {
        "models.py": '''
from pydantic import BaseModel
from typing import Optional

class User(BaseModel):
    """Database model for a standard system user."""
    id: int
    username: str
    email: str
    hashed_password: str

class Token(BaseModel):
    access_token: str
    token_type: str
''',
        "auth.py": '''
import time
from models import User, Token

SECRET_KEY = "super-secret-key"

def hash_password(password: str) -> str:
    """Hashes a plaintext password."""
    return f"hashed_{password}"

def authenticate_user(username: str, password: str) -> Optional[User]:
    """Authenticates a user via UUID session tokens. Needs refactoring to JWT."""
    # Dummy authentication logic
    if username == "admin" and password == "password":
        return User(id=1, username=username, email="admin@test.com", hashed_password=hash_password(password))
    return None
''',
        "main.py": '''
from fastapi import FastAPI, HTTPException
from auth import authenticate_user
from models import Token

app = FastAPI()

@app.post("/login")
def login(username: str, password: str) -> Token:
    """Login endpoint for the API."""
    user = authenticate_user(username, password)
    if not user:
        raise HTTPException(status_code=400, detail="Incorrect username or password")
    return Token(access_token="dummy-session-token", token_type="bearer")
'''
    }

    for filename, content in files.items():
        with open(repo_path / filename, 'w', encoding='utf-8') as f:
            f.write(content.strip())

    logger.info(f"Dummy repository created at {repo_path.absolute()}")

# ==============================================================================
# EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    REPO_DIR = "./dummy_repo"
    create_dummy_repository(REPO_DIR)

    analyzer = CodebaseAnalyzer()
    repo_metadata =[]

    for filename in os.listdir(REPO_DIR):
        if filename.endswith(".py"):
            filepath = os.path.join(REPO_DIR, filename)
            file_meta = analyzer.analyze_file(filepath)
            if file_meta:
                repo_metadata.append(asdict(file_meta))

    # Display the extracted FastCode-style AST metadata for auth.py
    logger.info("Successfully parsed repository. Example output for 'auth.py':")
    auth_meta = next(f for f in repo_metadata if "auth.py" in f['filepath'])
    print(json.dumps(auth_meta, indent=2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.1 MB/s eta 0:00:00
{
  "filepath": "./dummy_repo/auth.py",
  "imports": [
    {
      "module": "time",
      "names": [],
      "is_from_import": false
    },
    {
      "module": "models",
      "names": [
        "models",
        "User",
        "Token"
      ],
      "is_from_import": true
    }
  ],
  "classes": [],
  "functions": [
    {
      "name": "hash_password",
      "signature": "def hash_password(password: str) -> str:",
      "docstring": "Hashes a plaintext password.",
      "start_line": 6,
      "end_line": 8,
      "is_method": false
    },
    {
      "name": "authenticate_user",
      "signature": "def authenticate_user(username: str, password: str) -> Optional[User]:",
      "docstring": "Authenticates a user via UUID session tokens. Needs ref

In [ ]:
# ==============================================================================
# CELL 2: DYNAMIC MEMORY ENGINE (GRAPH + VECTOR DB)
# ==============================================================================

!pip install -q sentence-transformers

import networkx as nx
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import logging
from typing import List, Dict, Any, Tuple

logger = logging.getLogger("Dynamic_Memory")

class CodeGraphMemory:
    """
    Implements the LightRAG dual-level retrieval paradigm over a FastCode
    metadata structure. Fuses NetworkX (topology) with FAISS (semantics).
    """
    def __init__(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
        # Initialize Graph
        self.graph = nx.DiGraph()

        # Initialize Embedding Model
        logger.info(f"Loading embedding model: {embedding_model_name}")
        self.embedder = SentenceTransformer(embedding_model_name)
        self.embedding_dim = self.embedder.get_sentence_embedding_dimension()

        # Initialize FAISS Index (L2 distance)
        self.vector_index = faiss.IndexFlatL2(self.embedding_dim)

        # Mappings to bridge FAISS integer IDs to NetworkX string IDs
        self.id_to_node_name: Dict[int, str] = {}
        self.node_name_to_id: Dict[str, int] = {}
        self.current_id = 0

    def _add_to_vector_db(self, node_id: str, search_text: str):
        """Embeds text and adds it to the FAISS index."""
        if not search_text.strip():
            return

        embedding = self.embedder.encode([search_text], convert_to_numpy=True)
        self.vector_index.add(embedding)

        self.id_to_node_name[self.current_id] = node_id
        self.node_name_to_id[node_id] = self.current_id
        self.current_id += 1

    def ingest_metadata(self, repo_metadata: List[Dict[str, Any]]):
        """Builds the graph and vector DB from the FastCode AST metadata."""
        logger.info("Ingesting AST metadata into Graph and Vector DB...")

        for file_meta in repo_metadata:
            filepath = file_meta['filepath']
            filename = filepath.split('/')[-1]

            # 1. Add File Node
            self.graph.add_node(filepath, type='file', name=filename)

            # 2. Add Imports (Dependency Edges)
            for imp in file_meta['imports']:
                # Simple heuristic: link to the module name (e.g., 'models')
                target_module = f"{imp['module']}.py"
                self.graph.add_edge(filepath, target_module, relation='IMPORTS')

            # 3. Add Classes
            for cls in file_meta['classes']:
                cls_id = f"{filepath}::{cls['name']}"
                search_text = f"Class {cls['name']}. {cls['docstring'] or ''}"

                self.graph.add_node(cls_id, type='class', meta=cls)
                self.graph.add_edge(filepath, cls_id, relation='CONTAINS')
                self._add_to_vector_db(cls_id, search_text)

                # Add Methods inside Classes
                for method in cls['methods']:
                    method_id = f"{cls_id}::{method['name']}"
                    search_text = f"Method {method['signature']}. {method['docstring'] or ''}"

                    self.graph.add_node(method_id, type='method', meta=method)
                    self.graph.add_edge(cls_id, method_id, relation='CONTAINS')
                    self._add_to_vector_db(method_id, search_text)

            # 4. Add Module-Level Functions
            for func in file_meta['functions']:
                func_id = f"{filepath}::{func['name']}"
                search_text = f"Function {func['signature']}. {func['docstring'] or ''}"

                self.graph.add_node(func_id, type='function', meta=func)
                self.graph.add_edge(filepath, func_id, relation='CONTAINS')
                self._add_to_vector_db(func_id, search_text)

                # Semantic mapping heuristic: If 'User' is in signature, add dependency
                for cls_name in ["User", "Token"]:  # Hardcoded for our dummy repo test
                    if cls_name in func['signature']:
                        target_cls_id = f"./dummy_repo/models.py::{cls_name}"
                        self.graph.add_edge(func_id, target_cls_id, relation='REFERENCES')

        logger.info(f"Ingestion complete. Graph nodes: {self.graph.number_of_nodes()}, Edges: {self.graph.number_of_edges()}")

    def retrieve_with_context(self, query: str, top_k: int = 1) -> Dict[str, Any]:
        """
        The LightRAG dual-level retrieval algorithm.
        1. Semantic search via FAISS.
        2. Structural 1-hop expansion via NetworkX.
        """
        logger.info(f"Retrieving context for query: '{query}'")

        # 1. Semantic Vector Search
        query_embedding = self.embedder.encode([query], convert_to_numpy=True)
        distances, indices = self.vector_index.search(query_embedding, top_k)

        results = {}
        for idx in indices[0]:
            if idx == -1: continue # FAISS returns -1 if not enough results

            node_id = self.id_to_node_name[idx]
            primary_node = self.graph.nodes[node_id]

            # 2. NetworkX 1-Hop Expansion (Get neighbors)
            neighbors =[]
            for neighbor_id in self.graph.successors(node_id):
                edge_data = self.graph.get_edge_data(node_id, neighbor_id)
                if neighbor_id in self.graph.nodes: # Ensure node exists
                    neighbors.append({
                        "id": neighbor_id,
                        "relation": edge_data.get('relation', 'UNKNOWN'),
                        "type": self.graph.nodes[neighbor_id].get('type', 'file')
                    })

            # Reverse expansion (What contains this node?)
            for predecessor_id in self.graph.predecessors(node_id):
                edge_data = self.graph.get_edge_data(predecessor_id, node_id)
                neighbors.append({
                    "id": predecessor_id,
                    "relation": f"IS_{edge_data.get('relation', 'UNKNOWN')}_BY",
                    "type": self.graph.nodes[predecessor_id].get('type', 'file')
                })

            results[node_id] = {
                "target_node": primary_node,
                "1_hop_context": neighbors
            }

        return results

# ==============================================================================
# EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    # Ensure repo_metadata from Cell 1 exists
    try:
        repo_metadata
    except NameError:
        logger.error("repo_metadata not found. Please run Cell 1 first.")
    else:
        # Initialize Memory Engine
        memory_engine = CodeGraphMemory()

        # Build the Graph and Vector DB
        memory_engine.ingest_metadata(repo_metadata)

        # Simulate a developer intent query!
        query = "Refactor the authentication logic to use JWT tokens"

        # Execute LightRAG retrieval
        retrieval_results = memory_engine.retrieve_with_context(query, top_k=1)

        print("\n" + "="*50)
        print("RETRIEVAL ENGINE RESULTS")
        print("="*50)
        for node_id, data in retrieval_results.items():
            print(f"Target Hit: {node_id}")
            print(f"Node Data:  {data['target_node']['meta']['docstring']}")
            print("-" * 50)
            print("1-Hop Architectural Context:")
            for neighbor in data['1_hop_context']:
                print(f"  -> [{neighbor['relation']}] {neighbor['id']} ({neighbor['type']})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_7392/3539559550.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.embedder.get_sentence_embedding_dimension()



RETRIEVAL ENGINE RESULTS
Target Hit: ./dummy_repo/auth.py::authenticate_user
Node Data:  Authenticates a user via UUID session tokens. Needs refactoring to JWT.
--------------------------------------------------
1-Hop Architectural Context:
  -> [REFERENCES] ./dummy_repo/models.py::User (class)
  -> [IS_CONTAINS_BY] ./dummy_repo/auth.py (file)


In [ ]:
# ==============================================================================
# CELL 3: THE PROBABILISTIC ENGINE & BUDGETED ASSEMBLY
# ==============================================================================

# 1. Install LLM dependencies (accelerate for efficient GPU loading)
!pip install -q transformers accelerate torch

import torch
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Dict, Any

logger = logging.getLogger("Reasoning_Engine")

# ==============================================================================
# BUDGETED CONTEXT ASSEMBLER
# ==============================================================================
class ContextAssembler:
    """
    Takes the abstract metadata graph nodes and dynamically reads ONLY the
    required lines of code from disk, enforcing a strict token budget.
    """
    @staticmethod
    def extract_code_block(filepath: str, start_line: int, end_line: int) -> str:
        """Extracts exact lines from a source file."""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            # 0-indexed slice; inclusive of end_line
            return "".join(lines[start_line-1 : end_line])
        except Exception as e:
            logger.error(f"Failed to read {filepath}: {e}")
            return f"# [Error reading {filepath}]"

    @classmethod
    def assemble_prompt_context(cls, memory_engine: Any, retrieval_results: Dict[str, Any]) -> str:
        """Assembles the exact string context needed for the LLM."""
        logger.info("Assembling budgeted context from graph nodes...")
        context_blocks =[]

        for node_id, data in retrieval_results.items():
            target_meta = data['target_node']['meta']
            filepath = target_meta.get('filepath') or node_id.split('::')[0]

            # 1. Grab the target function body
            target_code = cls.extract_code_block(filepath, target_meta['start_line'], target_meta['end_line'])
            context_blocks.append(f"--- TARGET TO REFACTOR: {node_id} ---\n{target_code}\n")

            # 2. Grab the 1-Hop dependencies (e.g., the User class)
            for neighbor in data['1_hop_context']:
                n_id = neighbor['id']
                if '::' in n_id:  # It's a specific class or function, not a whole file
                    neighbor_node = memory_engine.graph.nodes[n_id]
                    if 'meta' in neighbor_node:
                        n_meta = neighbor_node['meta']
                        n_filepath = n_id.split('::')[0]
                        n_code = cls.extract_code_block(n_filepath, n_meta['start_line'], n_meta['end_line'])
                        context_blocks.append(f"--- 1-HOP DEPENDENCY: {n_id} [{neighbor['relation']}] ---\n{n_code}\n")

        return "\n".join(context_blocks)

# ==============================================================================
# LLM REASONING ENGINE
# ==============================================================================
class ReasoningEngine:
    """
    The probabilistic LLM wrapper. Loaded in bfloat16 for optimal A100 usage.
    """
    def __init__(self, model_id: str = "Qwen/Qwen2.5-Coder-7B-Instruct"):
        logger.info(f"Loading LLM {model_id} onto GPU. This may take 1-2 minutes...")

        # Load Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        # Load Model to A100 (cuda) using bfloat16 to maximize tensor core efficiency
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        logger.info("LLM successfully loaded and mapped to A100.")

    def generate_refactor(self, intent: str, context: str) -> str:
        """Generates the refactored code using the strict context."""
        logger.info("Drafting refactored code...")

        system_prompt = (
            "You are the reasoning engine of an AI-First IDE. "
            "You have been provided with exactly the architectural context required to fulfill the user's intent. "
            "Your output must contain ONLY the refactored Python code. Do not include markdown blocks, "
            "do not include explanations, and do not write test cases. Just output the raw, complete code for the requested target."
        )

        user_prompt = (
            f"User Intent: {intent}\n\n"
            f"Architectural Context:\n{context}\n\n"
            f"Refactor the target code to fulfill the intent."
        )

        messages =[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # Deterministic generation params for coding
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,  # Low temperature for strict coding
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        response = self.tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        return response.strip()

# ==============================================================================
# EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    # Ensure memory_engine and retrieval_results from Cell 2 exist
    try:
        memory_engine
        retrieval_results
    except NameError:
        logger.error("Dependencies from Cell 2 missing. Run Cell 2 first.")
    else:
        # 1. Assemble the exact code context
        assembled_context = ContextAssembler.assemble_prompt_context(memory_engine, retrieval_results)

        print("\n" + "="*50)
        print("BUDGETED CONTEXT ASSEMBLED (Sent to LLM)")
        print("="*50)
        print(assembled_context)

        # 2. Initialize the LLM (Requires GPU)
        llm_engine = ReasoningEngine()

        # 3. Generate the Code
        intent = "Refactor the authentication logic to use JWT tokens. Use the PyJWT library."
        drafted_code = llm_engine.generate_refactor(intent, assembled_context)

        print("\n" + "="*50)
        print("LLM GENERATED REFACTOR (Draft State)")
        print("="*50)
        print(drafted_code)


BUDGETED CONTEXT ASSEMBLED (Sent to LLM)
--- TARGET TO REFACTOR: ./dummy_repo/auth.py::authenticate_user ---
def authenticate_user(username: str, password: str) -> Optional[User]:
    """Authenticates a user via UUID session tokens. Needs refactoring to JWT."""
    # Dummy authentication logic
    if username == "admin" and password == "password":
        return User(id=1, username=username, email="admin@test.com", hashed_password=hash_password(password))
    return None

--- 1-HOP DEPENDENCY: ./dummy_repo/models.py::User [REFERENCES] ---
class User(BaseModel):
    """Database model for a standard system user."""
    id: int
    username: str
    email: str
    hashed_password: str




config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


LLM GENERATED REFACTOR (Draft State)
```python
import jwt
from datetime import datetime, timedelta
from typing import Optional
from .models import User

SECRET_KEY = 'your_secret_key'

def authenticate_user(username: str, password: str) -> Optional[str]:
    """Authenticates a user and returns a JWT token if successful."""
    # Dummy authentication logic
    if username == "admin" and password == "password":
        user = User(id=1, username=username, email="admin@test.com", hashed_password=hash_password(password))
        token = generate_jwt_token(user)
        return token
    return None

def generate_jwt_token(user: User) -> str:
    """Generates a JWT token for a given user."""
    payload = {
        'user_id': user.id,
        'exp': datetime.utcnow() + timedelta(hours=1)
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm='HS256')
    return token
```


In [ ]:
# ==============================================================================
# CELL 4: THE CLOSED-LOOP VERIFICATION GATE
# ==============================================================================

import json
import logging
from typing import Tuple, Dict, Any
import tree_sitter
import tree_sitter_python

logger = logging.getLogger("Verification_Gate")

class CodeVerifier:
    """
    Acts as the IDE's immune system. Traps syntax errors deterministically using
    an AST, and traps logical errors using an LLM-based ToT taxonomy check.
    """
    def __init__(self):
        self.language = tree_sitter.Language(tree_sitter_python.language())
        self.parser = tree_sitter.Parser(self.language)

    def clean_code(self, raw_code: str) -> str:
        """Heuristic cleaner for common LLM artifacts (like markdown)."""
        cleaned = raw_code.strip()
        if cleaned.startswith("```python"):
            cleaned = cleaned[9:]
        elif cleaned.startswith("```"):
            cleaned = cleaned[3:]

        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]

        return cleaned.strip()

    def verify_syntax(self, code: str) -> Tuple[bool, str]:
        """
        Uses tree-sitter to deterministically check if the code is valid Python.
        Returns (is_valid, error_message).
        """
        logger.info("Running deterministic AST syntax check...")
        tree = self.parser.parse(code.encode('utf8'))

        # Walk the AST looking for ERROR nodes
        def contains_error(node) -> bool:
            if node.type == 'ERROR':
                return True
            for child in node.children:
                if contains_error(child):
                    return True
            return False

        if contains_error(tree.root_node):
            return False, "AST Parsing Failed: The generated code contains invalid Python syntax."

        return True, "Syntax valid."

    def verify_logic(self, code: str, intent: str, llm_engine: Any) -> Tuple[bool, str]:
        """
        Implements the Logical Error classification from the academic paper.
        Forces the LLM to use Chain-of-Thought to check for logical failures.
        """
        logger.info("Running probabilistic logical error check...")

        system_prompt = (
            "You are a rigorous code reviewer checking for Logical Errors in a refactoring task. "
            "A Logical Error occurs when code compiles but operates against the programmer's intent. "
            "Check specifically for:\n"
            "1. (B) Output Error: Does the function return the wrong format or type?\n"
            "2. (I) Function Error: Are the arguments or return statements fundamentally flawed based on the intent?\n\n"
            "Analyze the code step-by-step. Then output a JSON object with two keys: "
            "'is_logically_sound' (boolean) and 'reasoning' (string)."
        )

        user_prompt = (
            f"Original Intent: {intent}\n\n"
            f"Drafted Code:\n{code}\n\n"
            f"Perform your analysis and return the JSON."
        )

        messages =[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        text = llm_engine.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = llm_engine.tokenizer([text], return_tensors="pt").to(llm_engine.model.device)

        outputs = llm_engine.model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            pad_token_id=llm_engine.tokenizer.eos_token_id
        )

        response = llm_engine.tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

        # Attempt to parse the JSON response
        try:
            # Find JSON block if wrapped in markdown
            if "```json" in response:
                json_str = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                json_str = response.split("```")[1].strip()
            else:
                json_str = response.strip()

            result = json.loads(json_str)
            return result.get('is_logically_sound', False), result.get('reasoning', "No reasoning provided.")
        except Exception as e:
            logger.error(f"Failed to parse logic check JSON: {str(e)}\nRaw output: {response}")
            return False, "Failed to validate logic."

# ==============================================================================
# EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    try:
        drafted_code
        llm_engine
        intent
    except NameError:
        logger.error("Dependencies missing. Please ensure Cell 3 completed successfully.")
    else:
        verifier = CodeVerifier()

        print("\n" + "="*50)
        print("STAGE 1: RAW DRAFT VERIFICATION")
        print("="*50)
        # We test the raw code (with the markdown tags) to prove it fails the AST
        is_syntax_valid, syntax_msg = verifier.verify_syntax(drafted_code)
        print(f"AST Syntax Check (Raw Code): {'PASS' if is_syntax_valid else 'FAIL'}")
        print(f"Message: {syntax_msg}")

        print("\n" + "="*50)
        print("STAGE 2: AUTO-CORRECTION & RE-VERIFICATION")
        print("="*50)
        cleaned_code = verifier.clean_code(drafted_code)
        is_syntax_valid, syntax_msg = verifier.verify_syntax(cleaned_code)
        print(f"AST Syntax Check (Cleaned Code): {'PASS' if is_syntax_valid else 'FAIL'}")

        if is_syntax_valid:
            print("\n" + "="*50)
            print("STAGE 3: LOGICAL ERROR ANALYSIS (ToT)")
            print("="*50)
            is_logic_valid, logic_msg = verifier.verify_logic(cleaned_code, intent, llm_engine)
            print(f"Logical Check: {'PASS' if is_logic_valid else 'FAIL'}")
            print(f"Reasoning:\n{logic_msg}")

            if is_logic_valid:
                print("\n" + "="*50)
                print("FINAL RESULT: CODE APPROVED FOR IDE INSERTION")
                print("="*50)
                print(cleaned_code)
            else:
                print("\nFINAL RESULT: CODE REJECTED BY LOGIC GATE.")
        else:
            print("\nFINAL RESULT: CODE REJECTED BY SYNTAX GATE.")


STAGE 1: RAW DRAFT VERIFICATION
AST Syntax Check (Raw Code): FAIL
Message: AST Parsing Failed: The generated code contains invalid Python syntax.

STAGE 2: AUTO-CORRECTION & RE-VERIFICATION
AST Syntax Check (Cleaned Code): PASS

STAGE 3: LOGICAL ERROR ANALYSIS (ToT)
Logical Check: FAIL
Reasoning:
The function `authenticate_user` contains a dummy authentication logic that hardcodes the username and password. This is not secure and does not reflect the intended functionality of authenticating users based on their credentials. Additionally, the `generate_jwt_token` function uses a hardcoded secret key, which should be securely managed and not exposed.

FINAL RESULT: CODE REJECTED BY LOGIC GATE.
